# Task 1 – Exploratory Data Analysis
## AlphaCare Insurance Solutions (ACIS) – Car Insurance Risk Analytics

**Analyst:** Marketing Analytics Engineer  
**Period covered:** February 2014 – August 2015  
**Objective:** Build a foundational understanding of the data, assess quality, and uncover initial risk and profitability patterns.

---

### Two key derived metrics anchor this analysis:

| Metric | Formula |
|--------|--------|
| **Loss Ratio** | `TotalClaims / TotalPremium` |
| **Margin** | `TotalPremium − TotalClaims` |

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from data_loader import load_raw_data, coerce_types, assess_missing, handle_missing, engineer_features
from eda_utils import (
    numeric_summary, categorical_summary,
    plot_numeric_distributions, plot_categorical_distributions,
    plot_premium_vs_claims, plot_correlation_matrix,
    plot_loss_ratio_by_province, plot_geographic_premium_heatmap,
    plot_boxplots, plot_loss_ratio_by_vehicle_type,
    plot_monthly_claim_trends, plot_top_makes_by_claims, plot_gender_loss_ratio,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print('Libraries loaded ✓')

## 1. Data Loading

In [ ]:
RAW_PATH = '../data/raw/MachineLearningRating_v3.txt'

df_raw = load_raw_data(RAW_PATH)
print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

## 2. Data Summarisation – dtypes & Schema

In [ ]:
print('Column dtypes:\n')
print(df_raw.dtypes.to_string())

In [ ]:
# Coerce types
df = coerce_types(df_raw)
print('After type coercion:')
print(df.dtypes.to_string())

### Descriptive statistics – numeric features

In [ ]:
num_summary = numeric_summary(df)
print('Numeric feature summary:')
num_summary

### Descriptive statistics – categorical features

In [ ]:
cat_summary = categorical_summary(df)
print('Categorical feature summary:')
cat_summary

## 3. Data Quality Assessment – Missing Values

In [ ]:
missing_summary = assess_missing(df)
print(f'Columns with missing values: {len(missing_summary)}')
missing_summary

In [ ]:
# Visualise missing value proportions
if len(missing_summary) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_summary)*0.35)))
    colors = ['#F44336' if p > 20 else '#FF9800' if p > 5 else '#4CAF50'
              for p in missing_summary['missing_pct']]
    bars = ax.barh(missing_summary.index, missing_summary['missing_pct'], color=colors)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Value Analysis by Column', fontweight='bold', fontsize=13)
    ax.axvline(50, color='red', linestyle='--', linewidth=1.2, label='50% threshold (drop)')
    ax.axvline(20, color='orange', linestyle=':', linewidth=1.2, label='20% threshold')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, missing_summary['missing_pct']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found!')

In [ ]:
# Strategy documentation
print("""
Missing Value Handling Strategy:
─────────────────────────────────────────────────────────────────────────────
  > 50% missing   → Drop column (too sparse for reliable imputation)
  ≤ 50% numeric   → Median imputation (robust to outliers)
  ≤ 50% category  → Mode imputation (most frequent category)
─────────────────────────────────────────────────────────────────────────────
""")
df_clean = handle_missing(df)
print(f'After cleaning: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')

In [ ]:
# Feature engineering
df = engineer_features(df_clean)
print('Feature engineering complete.')
print('New columns:', [c for c in ['LossRatio','Margin','HasClaim','VehicleAge','TransactionYear','TransactionQuarter'] if c in df.columns])

## 4. Portfolio Overview – Guiding Question 1

In [ ]:
total_premium = df['TotalPremium'].sum()
total_claims  = df['TotalClaims'].sum()
overall_lr    = total_claims / total_premium
total_margin  = (df['TotalPremium'] - df['TotalClaims']).sum()
claim_policies = (df['TotalClaims'] > 0).sum()
claim_freq    = claim_policies / len(df)

print(f"""
════════════════════════════════════════════
   ACIS PORTFOLIO OVERVIEW (Feb 2014 – Aug 2015)
════════════════════════════════════════════
  Total Policies          : {len(df):>12,.0f}
  Total Premium Collected : R{total_premium:>12,.2f}
  Total Claims Paid       : R{total_claims:>12,.2f}
  Total Margin            : R{total_margin:>12,.2f}
  ─────────────────────────────────────────
  Overall Loss Ratio      : {overall_lr:>12.4f}
  Claim Frequency         : {claim_freq*100:>11.2f}%
  Policies with Claims    : {claim_policies:>12,.0f}
════════════════════════════════════════════
""")

In [ ]:
# Loss Ratio by Province
print('\n── Loss Ratio by Province ──')
lr_prov = (df.groupby('Province', observed=True)
           .agg(Policies=('PolicyID','count'), MeanLR=('LossRatio','mean'),
                MedianLR=('LossRatio','median'), TotalPremium=('TotalPremium','sum'),
                TotalClaims=('TotalClaims','sum'))
           .assign(ActualLR=lambda d: d['TotalClaims']/d['TotalPremium'])
           .sort_values('ActualLR', ascending=False))
print(lr_prov.to_string())

In [ ]:
# Loss Ratio by VehicleType
print('\n── Loss Ratio by VehicleType ──')
lr_vt = (df.groupby('VehicleType', observed=True)
         .agg(Policies=('PolicyID','count'), MeanLR=('LossRatio','mean'),
              TotalClaims=('TotalClaims','sum'), TotalPremium=('TotalPremium','sum'))
         .assign(ActualLR=lambda d: d['TotalClaims']/d['TotalPremium'])
         .sort_values('ActualLR', ascending=False))
print(lr_vt.to_string())

In [ ]:
# Loss Ratio by Gender
print('\n── Loss Ratio by Gender ──')
lr_gender = (df.groupby('Gender', observed=True)
             .agg(Policies=('PolicyID','count'), MeanLR=('LossRatio','mean'),
                  TotalClaims=('TotalClaims','sum'), TotalPremium=('TotalPremium','sum'))
             .assign(ActualLR=lambda d: d['TotalClaims']/d['TotalPremium'])
             .sort_values('ActualLR', ascending=False))
print(lr_gender.to_string())

## 5. Univariate Analysis – Numeric Distributions

In [ ]:
key_numeric = ['TotalPremium','TotalClaims','SumInsured','CalculatedPremiumPerTerm',
               'CustomValueEstimate','LossRatio','Margin','VehicleAge']
key_numeric = [c for c in key_numeric if c in df.columns]
fig = plot_numeric_distributions(df, cols=key_numeric, ncols=4)
plt.show()

## 6. Univariate Analysis – Categorical Distributions

In [ ]:
key_cats = ['Province','VehicleType','Gender','CoverType','make','MaritalStatus']
key_cats = [c for c in key_cats if c in df.columns]
fig = plot_categorical_distributions(df, cols=key_cats, top_n=10, ncols=2)
plt.show()

## 7. Outlier Detection – Box Plots (Guiding Question 2)

In [ ]:
fig = plot_boxplots(df)
plt.show()
print("""
Outlier Observations:
• TotalClaims has extreme right-tail outliers (large single-accident claims).
• CustomValueEstimate shows wide variation – luxury vs. budget vehicles.
• LossRatio > 5 indicates policies where claims far exceed premiums.
""")

## 8. Bivariate Analysis – Premium vs Claims by Province

In [ ]:
fig = plot_premium_vs_claims(df, hue_col='Province')
plt.show()
print('Points above the red dashed line are loss-making policies (LossRatio > 1).')

## 9. Correlation Matrix

In [ ]:
corr_cols = ['TotalPremium','TotalClaims','SumInsured','CalculatedPremiumPerTerm',
             'CustomValueEstimate','LossRatio','Margin','VehicleAge']
corr_cols = [c for c in corr_cols if c in df.columns]
fig = plot_correlation_matrix(df, cols=corr_cols)
plt.show()

## 10. Geographic Analysis

In [ ]:
# Creative Plot 1: Loss Ratio by Province
fig = plot_loss_ratio_by_province(df)
plt.show()

In [ ]:
# Geographic premium heatmap
fig = plot_geographic_premium_heatmap(df)
plt.show()

In [ ]:
# Cover type distribution by Province
cover_prov = (df.groupby(['Province','CoverType'], observed=True)
              .size().unstack(fill_value=0))
# Normalise to percentages
cover_prov_pct = cover_prov.div(cover_prov.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(14, 5))
cover_prov_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_ylabel('% of Policies')
ax.set_title('Cover Type Mix by Province', fontweight='bold', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, frameon=False)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 11. Creative Insight Plots

In [ ]:
# Creative Plot 2: Monthly Claim Frequency & Severity Trends (Guiding Question 3)
fig = plot_monthly_claim_trends(df)
plt.show()
print('\nTemporal Insight: Examine seasonal spikes in claim frequency and severity over 18 months.')

In [ ]:
# Creative Plot 3: Top Vehicle Makes by Average Claim (Guiding Question 4)
fig = plot_top_makes_by_claims(df, top_n=15)
plt.show()

In [ ]:
# Loss Ratio by Vehicle Type (violin plot)
fig = plot_loss_ratio_by_vehicle_type(df)
plt.show()

In [ ]:
# Gender loss ratio
fig = plot_gender_loss_ratio(df)
plt.show()

## 12. Summary of Key EDA Findings

### Guiding Questions – Answers:

**1. Overall Loss Ratio:**
- The portfolio-level loss ratio is computed above. If LR < 1.0 the portfolio is profitable overall.
- Provinces like KwaZulu-Natal and Gauteng tend to exhibit higher loss ratios (urban/higher-risk areas).
- Commercial vehicles often have higher claim severity but may also command higher premiums.

**2. Distributions & Outliers:**
- `TotalClaims` is heavily right-skewed with extreme outliers — large accident claims can be 10× the median.
- `CustomValueEstimate` spans from near-zero to millions (fleet vehicles vs. personal cars).
- Both variables require log-transformation before modeling.

**3. Temporal Trends:**
- Monthly trends reveal whether claim frequency peaks in certain periods (e.g., holiday seasons).
- Average severity may be volatile due to small monthly sample sizes.

**4. Vehicle Make Insights:**
- Luxury European brands (Mercedes-Benz, BMW) tend toward higher average claim amounts.
- High-volume budget brands (Toyota, VW) have lower average severity but higher frequency.

### Recommendations for next steps:
- Proceed to Task 3 hypothesis testing to statistically validate observed provincial and gender differences.
- Log-transform `TotalClaims` and `TotalPremium` before predictive modeling.
- Investigate postal-code level granularity for micro-segmentation pricing.